# Data Ingestion
Download the four Amazon review TSV files from Kaggle, verify them, and save a data inventory.

## 1. Setup

In [4]:
import os
import kagglehub

os.environ["KAGGLEHUB_CACHE"] = "./data"

DATASET = "cynthiarempel/amazon-us-customer-reviews-dataset"

FILES = [
    "amazon_reviews_us_Video_DVD_v1_00.tsv",
    "amazon_reviews_us_Music_v1_00.tsv",
    "amazon_reviews_us_Digital_Ebook_Purchase_v1_01.tsv",
    "amazon_reviews_us_Books_v1_02.tsv",
]

EXPECTED_HEADER = [
    "marketplace", "customer_id", "review_id", "product_id", "product_parent",
    "product_title", "product_category", "star_rating", "helpful_votes",
    "total_votes", "vine", "verified_purchase", "review_headline",
    "review_body", "review_date",
]

## 2. Download (skips files that are already downloaded)

In [5]:
paths = {}
for name in FILES:
    print(f"Downloading {name} ...")
    paths[name] = kagglehub.dataset_download(DATASET, path=name)
    print("  saved to:", paths[name])

print("\nAll files:")
for name, p in paths.items():
    size_gb = os.path.getsize(p) / 1e9
    print(f"{name}: {size_gb:.2f} GB -> {p}")

  saved to: ./data/datasets/cynthiarempel/amazon-us-customer-reviews-dataset/versions/9/amazon_reviews_us_Video_DVD_v1_00.tsv
  saved to: ./data/datasets/cynthiarempel/amazon-us-customer-reviews-dataset/versions/9/amazon_reviews_us_Music_v1_00.tsv
  saved to: ./data/datasets/cynthiarempel/amazon-us-customer-reviews-dataset/versions/9/amazon_reviews_us_Digital_Ebook_Purchase_v1_01.tsv
  saved to: ./data/datasets/cynthiarempel/amazon-us-customer-reviews-dataset/versions/9/amazon_reviews_us_Books_v1_02.tsv

All files:
amazon_reviews_us_Video_DVD_v1_00.tsv: 3.71 GB -> ./data/datasets/cynthiarempel/amazon-us-customer-reviews-dataset/versions/9/amazon_reviews_us_Video_DVD_v1_00.tsv
amazon_reviews_us_Music_v1_00.tsv: 3.67 GB -> ./data/datasets/cynthiarempel/amazon-us-customer-reviews-dataset/versions/9/amazon_reviews_us_Music_v1_00.tsv
amazon_reviews_us_Digital_Ebook_Purchase_v1_01.tsv: 3.22 GB -> ./data/datasets/cynthiarempel/amazon-us-customer-reviews-dataset/versions/9/amazon_reviews_us_Di

## 3. Verify headers, sizes, and row counts

In [6]:
def count_lines(path):
    """Fast line count (binary mode), minus 1 for the header."""
    n = 0
    with open(path, "rb") as f:
        while block := f.read(1 << 24):
            n += block.count(b"\n")
    return n - 1

results = []
for category, path in paths.items():
    with open(path, encoding="utf-8", errors="replace") as f:
        header = f.readline().rstrip("\n").split("\t")
    assert header == EXPECTED_HEADER, f"{category}: unexpected header {header}"

    size_gb = os.path.getsize(path) / 1e9
    rows = count_lines(path)
    results.append((category, size_gb, rows))
    print(f"{category}: {size_gb:.2f} GB | {rows:,} rows | header OK")

total_gb = sum(r[1] for r in results)
total_rows = sum(r[2] for r in results)
print(f"\nTOTAL: {total_gb:.2f} GB | {total_rows:,} rows")

amazon_reviews_us_Video_DVD_v1_00.tsv: 3.71 GB | 5,069,140 rows | header OK
amazon_reviews_us_Music_v1_00.tsv: 3.67 GB | 4,751,577 rows | header OK
amazon_reviews_us_Digital_Ebook_Purchase_v1_01.tsv: 3.22 GB | 5,101,693 rows | header OK
amazon_reviews_us_Books_v1_02.tsv: 3.24 GB | 3,105,520 rows | header OK

TOTAL: 13.84 GB | 18,027,930 rows
